# Category

**Target: avoid hybrid categories**

## Step 1. See outliers in "category"

### Load data

In [2]:
import pandas as pd

cases = pd.read_csv("cases.csv")

### Find the inconsistent category labels

In [3]:
cases["category"].value_counts()

,count
category,
Incident,71
Near Miss,56
Security,34
Safety,27
Quality,4
Reliability,1
Operational Efficiency,1
Quality / Security,1
Safety – Near Miss,1


### Filter the problematic categories

The dash may be different (- vs –), so we match both.

In [ ]:
problem_cases = cases[
    cases["category"].str.contains(
        r"quality\s*/\s*security|safety\s*[–-]\s*near miss",
        case=False,
        na=False,
        regex=True
    )
]

problem_cases[["case_id", "title", "category"]]

,case_id,title,category
164,165,Workflow Bypass Misconfiguration Allowed,Quality / Security
185,186,Hidden Hydraulic Accumulator Pressure,Safety – Near Miss


### Print everything including actions

In [ ]:
for _, row in problem_cases.iterrows():
    print("="*80)
    print(f"CASE ID: {row['case_id']}")
    print(f"TITLE: {row['title']}")
    print(f"CATEGORY: {row['category']}")
    print("="*80)

    for col in ["what_happened","root_causes","causal_factors","what_went_well","lessons","actions_raw"]:
        print(f"\n--- {col.upper()} ---\n")
        print(row[col])

    print("\n\n")

CASE ID: 165
TITLE: Workflow Bypass Misconfiguration Allowed
CATEGORY: Quality / Security

--- WHAT_HAPPENED ---

During routine scheduling in a digital work‑management platform, a newly deployed automation rule
was intended to streamline low‑risk task approvals. However, an incorrect condition in the rule logic
caused several work orders—including one involving equipment isolation—to skip the mandatory
technical review stage.
An operator noticed that a work order requesting equipment shutdown appeared directly in the
execution queue without the usual engineering sign‑off. Upon investigation, the planning team
discovered that the workflow engine had incorrectly flagged the job as “administrative,” allowing it to
bypass verification steps. The bypass was immediately disabled, and all affected tasks were paused
and manually reviewed. No field work had begun.

--- ROOT_CAUSES ---

● Automation rule was designed and deployed without structured testing or peer review.
● Workflow logic relie

## Step 2. Determine the true categories (with reasoning)

### CASE 165

**Current label: Quality / Security**

**What it actually is:**

Primary domain: Security / Control Integrity (Operational Governance)

**Why:**

This case involves:

	•	workflow automation bypass
	•	approval controls skipped
	•	change-management governance
	•	system configuration risk
	•	access/control safeguards

This is NOT quality (no product/service quality impact).

It is about control security & governance risk.

Strong indicators:

	•	bypassed approval workflow
	•	automation logic vulnerability
	•	governance & audit controls
	•	system monitoring & alerts

**This fits Security category.**


### CASE 186

**Current label: Safety – Near Miss**

**What it actually is:**

Primary domain: Near Miss

**Why:**

Your taxonomy separates:

	•	Incident
	•	Near Miss
	•	Safety

This case:

	•	had stored energy release
	•	posed serious injury risk
	•	stopped before harm
	•	no injury occurred

**This fits Near Miss category.**

## Step 3. Recategorize these cases

In [ ]:
cases.loc[cases["case_id"] == 165, "category"] = "Security"
cases.loc[cases["case_id"] == 186, "category"] = "Near Miss"

### Verify cleaned categories

In [ ]:
cases["category"].value_counts()

,count
category,
Incident,71
Near Miss,57
Security,35
Safety,27
Quality,4
Reliability,1
Operational Efficiency,1


# Location

## Step 1. Print ALL unique location values and show counts

In [ ]:
cases["location"].value_counts()

,count
location,
Canada,85
Vancouver,61
Working from Home,16
Trinidad & Tobago,13
USA,11
"Vancouver, Canada",3
Chile,3
New Zealand,1
Working from Home (Canada),1


## Step 2. Country Mapping

Re-categorize the locations so that they are all country-level.

### The function

In [ ]:
def map_country(location):
    if pd.isna(location):
        return location

    loc = location.strip().lower()

    # --- CANADA ---
    if loc in ["canada", "vancouver", "vancouver, canada"]:
        return "Canada"

    if loc == "working from home (canada)":
        return "Canada"

    # --- REMOTE WORK ---
    if loc == "working from home":
        return "Remote"

    # --- OTHER COUNTRIES ---
    if loc == "usa":
        return "USA"

    if loc == "trinidad & tobago":
        return "Trinidad & Tobago"

    if loc == "chile":
        return "Chile"

    if loc == "new zealand":
        return "New Zealand"

    if loc == "brussels":
        return "Belgium"

    if loc == "egypt":
        return "Egypt"

    # fallback (for future data)
    return location

### Apply the mapping

In [ ]:
cases["country"] = cases["location"].apply(map_country)

cases[["location","country"]].drop_duplicates().sort_values("country")

,location,country
143,Brussels,Belgium
0,Canada,Canada
5,"Vancouver, Canada",Canada
9,Vancouver,Canada
44,Working from Home (Canada),Canada
18,Chile,Chile
184,Egypt,Egypt
30,New Zealand,New Zealand
13,Working from Home,Remote
8,Trinidad & Tobago,Trinidad & Tobago


## Step 3. Verify results

In [ ]:
cases["country"].value_counts()

,count
country,
Canada,150
Remote,16
Trinidad & Tobago,13
USA,11
Chile,3
New Zealand,1
Belgium,1
Egypt,1


# Setting

## Step 1. Create top-level setting field

This handles:

* en dash (–)
* hyphen (-)
* inconsistent spacing

In [ ]:
def extract_primary_setting(setting):
    if pd.isna(setting):
        return setting

    # split on dash variants
    primary = setting.split("–")[0].split("-")[0].strip()

    return primary

cases["setting_primary"] = cases["setting"].apply(extract_primary_setting)

### View Results

In [ ]:
cases["setting_primary"].value_counts()

,count
setting_primary,
Operations,31
Process Unit,22
Working from Home,16
Corporate Office,11
Utilities Area,10
...,...
Utilities and Process Isolation,1
Utilities / Process Controls,1
Utilities / Hydraulic Actuation System,1


In [ ]:
# show all unique "setting_primary"
unique_setting_primary = sorted(cases["setting_primary"].dropna().unique())

for sett in unique_setting_primary:
    print(sett)

Additives Bay
Analytical Laboratory
Café
Central control room and remote cloud monitoring platform
Chemical Reactor Area
Chemical Transfer Bay
Chemical processing unit
Chemical unloading bay at a mid
Compression unit
Construction Zone
Construction/Laydown Area
Corporate IT Environment
Corporate Office
Corporate Operations
Corporate Operations Hub
Corporate data platform and order
Data Center
Digital Work‑Management & Automation Scheduling Systems
Digital Work‑Management System
Digital operations
Effluent Treatment Unit
Electrical Equipment Room
Electrical building
Equipment Startup Area
Facility Perimeter Road & Logistics Entry Gate
Field maintenance scheduling system
Furnace Duct
Gas Compression Area
General Processing Area
Heat‑Exchanger Access Chamber
Industrial Electrical Mezzanine
Laboratory Waste Handling Area
Main Facility Entrance & Process Control Corridor
Maintenance & Work Management System
Maintenance Area
Maintenance Bay
Maintenance Workshop & Loading Bay
Mechanical Area
M

## Step 2. Mapping with a clean, single-layer taxonomy

### Re-categorize

In [ ]:
import re

# Single-layer taxonomy mapping (based on the exact unique values)
SETTING_PRIMARY_TO_BUCKET = {
    # Industrial Operations (core process / production)
    "Additives Bay": "Industrial Operations",
    "Chemical Reactor Area": "Industrial Operations",
    "Chemical Transfer Bay": "Industrial Operations",
    "Chemical processing unit": "Industrial Operations",
    "Chemical unloading bay at a mid": "Industrial Operations",
    "Compression unit": "Industrial Operations",
    "Gas Compression Area": "Industrial Operations",
    "General Processing Area": "Industrial Operations",
    "Process Area": "Industrial Operations",
    "Process Platform": "Industrial Operations",
    "Process Pump Bay": "Industrial Operations",
    "Process Unit": "Industrial Operations",
    "Process Vessel": "Industrial Operations",
    "Production Unit": "Industrial Operations",
    "Reactor Vessel": "Industrial Operations",
    "Reactor sump pit": "Industrial Operations",
    "Pump Skid": "Industrial Operations",
    "Storage Tank": "Industrial Operations",
    "Transfer Pump Bay": "Industrial Operations",
    "Equipment Startup Area": "Industrial Operations",
    "Furnace Duct": "Industrial Operations",
    "Heat-Exchanger Access Chamber": "Industrial Operations",
    "Heat-Exchanger Access Chamber": "Industrial Operations",  # just in case
    "Overhead Column Platform": "Industrial Operations",
    "Plant Shutdown": "Industrial Operations",

    # Utilities & Infrastructure
    "Utilities": "Utilities & Infrastructure",
    "Utilities Area": "Utilities & Infrastructure",
    "Utilities area": "Utilities & Infrastructure",
    "Utilities/Process Area": "Utilities & Infrastructure",
    "Utilities & Mechanical Corridor": "Utilities & Infrastructure",
    "Utilities Substation": "Utilities & Infrastructure",
    "Utilities Control Network": "Utilities & Infrastructure",
    "Utilities / Process Controls": "Utilities & Infrastructure",
    "Utilities / Hydraulic Actuation System": "Utilities & Infrastructure",
    "Utilities and Process Isolation": "Utilities & Infrastructure",
    "Process Utilities Area": "Utilities & Infrastructure",
    "Effluent Treatment Unit": "Utilities & Infrastructure",
    "Wastewater Treatment Area": "Utilities & Infrastructure",
    "Water Treatment Area": "Utilities & Infrastructure",
    "Overhead Utility Walkway": "Utilities & Infrastructure",

    # Maintenance & Mechanical Work Areas
    "Maintenance Area": "Maintenance & Mechanical",
    "Maintenance Bay": "Maintenance & Mechanical",
    "Maintenance Workshop & Loading Bay": "Maintenance & Mechanical",
    "Mechanical Area": "Maintenance & Mechanical",
    "Mechanical Maintenance": "Maintenance & Mechanical",
    "Mechanical Maintenance Area": "Maintenance & Mechanical",
    "Mechanical Workshop": "Maintenance & Mechanical",
    "Maintenance & Work Management System": "Maintenance & Mechanical",
    "Field maintenance scheduling system": "Maintenance & Mechanical",

    # Electrical & Power Systems
    "Electrical Equipment Room": "Electrical & Power Systems",
    "Electrical building": "Electrical & Power Systems",
    "Industrial Electrical Mezzanine": "Electrical & Power Systems",
    "Office Electrical/Mechanical Room": "Electrical & Power Systems",

    # Laboratories & Testing
    "Analytical Laboratory": "Laboratories & Testing",
    "Quality Control Laboratory": "Laboratories & Testing",
    "Laboratory Waste Handling Area": "Laboratories & Testing",

    # Operations Control & Digital Systems
    "Central control room and remote cloud monitoring platform": "Operations Control & Digital Systems",
    "Digital operations": "Operations Control & Digital Systems",
    "Digital Work-Management & Automation Scheduling Systems": "Operations Control & Digital Systems",
    "Digital Work-Management & Automation Scheduling Systems": "Operations Control & Digital Systems",
    "Digital Work-Management System": "Operations Control & Digital Systems",
    "Digital Work-Management System": "Operations Control & Digital Systems",
    "Process Control Network": "Operations Control & Digital Systems",
    "Corporate data platform and order": "Operations Control & Digital Systems",
    "Data Center": "Operations Control & Digital Systems",
    "Operations Control Wing": "Operations Control & Digital Systems",
    "Operations IT Workspace": "Operations Control & Digital Systems",
    "Corporate IT Environment": "Operations Control & Digital Systems",

    # Corporate / Office / Administrative
    "Corporate Office": "Corporate / Office / Administrative",
    "Corporate Operations": "Corporate / Office / Administrative",
    "Corporate Operations Hub": "Corporate / Office / Administrative",
    "Operations Support Office": "Corporate / Office / Administrative",
    "Office Mechanical Room": "Corporate / Office / Administrative",

    # Construction & Temporary Work Zones
    "Construction Zone": "Construction & Temporary Work Zones",
    "Construction/Laydown Area": "Construction & Temporary Work Zones",

    # Logistics, Access & Perimeter Areas
    "Facility Perimeter Road & Logistics Entry Gate": "Logistics, Access & Perimeter Areas",
    "Main Facility Entrance & Process Control Corridor": "Logistics, Access & Perimeter Areas",
    "Warehouse & Controlled Operations Corridor": "Logistics, Access & Perimeter Areas",

    # Remote & Public Work Environments
    "Remote Work": "Remote & Public Work Environments",
    "Working From Home": "Remote & Public Work Environments",
    "Working from Home": "Remote & Public Work Environments",
    "Café": "Remote & Public Work Environments",
    "Public Café": "Remote & Public Work Environments",

    # Operations (generic catch from your list)
    "Operations": "Industrial Operations",
    "Operations Building": "Industrial Operations",
    "Operations building": "Industrial Operations",
    "Operations & Maintenance Area": "Industrial Operations",
    "Operations & Maintenance Building": "Industrial Operations",
}

### Apply mapping

In [ ]:
cases["setting_bucket"] = cases["setting_primary"].map(SETTING_PRIMARY_TO_BUCKET).fillna("Other / Review")

### View Result

In [ ]:
cases["setting_bucket"].value_counts()

,count
setting_bucket,
Industrial Operations,83
Utilities & Infrastructure,30
Remote & Public Work Environments,29
Corporate / Office / Administrative,15
Maintenance & Mechanical,11
Operations Control & Digital Systems,11
Electrical & Power Systems,5
Other / Review,4
Laboratories & Testing,3


# Export cleaned data

In [ ]:
cases.to_csv("cases_cleaned.csv", index=False)